# Colab: offline productivity LoRA training

Run the cells in order. Start with **Runtime → Change runtime type → GPU** and a
Python 3.11/3.12 runtime (the documented 2026.07 image uses Python 3.12).
This notebook runs a **two-step smoke test by default**, not a final experiment.
It does not mark research gates passed or execute phone benchmarks.

[Complete guide](https://github.com/DenysMenfredy/masters-sllms-research/blob/main/docs/colab.md)
| [Research repository](https://github.com/DenysMenfredy/masters-sllms-research)

Public code/data/model downloads and Drive authorization happen only when you run
these cells. Colab training is online; the offline requirement applies to the later
Android deployment. No Hugging Face token is needed for the public candidates.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/DenysMenfredy/masters-sllms-research.git"
REPOSITORY_REF = "main"  # Use the saved exact code commit for a resumed/final experiment.
REPO = Path("/content/masters-sllms-research")
PYTHON = Path("/content/sllm-cu126-venv/bin/python")
CANDIDATE = "qwen3_0_6b"  # Or qwen3_5_0_8b; both still require training-to-phone pilots.
SEED = 17
RECIPE = "configs/training/colab_smoke.yaml"
USE_QLORA = False  # Memory fallback only. LoRA is the primary recipe.
MOUNT_DRIVE = True
RUN_ID = "qwen3-lora-smoke-seed17-001"  # New ID per independent run; retain it to resume.
RESUME = False
DO_MERGE = False  # Turn on after inspecting a completed smoke run.

os.environ["HF_HOME"] = "/content/.cache/huggingface"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"


def run(command):
    child_env = os.environ.copy()
    child_env.pop("PYTHONPATH", None)
    child_env["PYTHONNOUSERSITE"] = "1"
    subprocess.run([str(item) for item in command], cwd=REPO, env=child_env, check=True)

## 1. Fetch code and resolve a commit

The default resolves `main` once. For reproducibility, record the printed commit.
On reconnect, set `REPOSITORY_REF` to the `code_revision` in the saved `run.json`.
Setup lists local changes and saves them, including untracked files, in a named
Git stash before checking out the requested revision. It prints the backup hash
and inspection/restoration commands. Saved edits are excluded from this run;
review and deliberately restore any intended recipe changes before training.
The stash lives on this Colab VM; export wanted edits before deleting the runtime.


In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPO)], check=True)
run(["git", "fetch", "--depth", "1", "origin", REPOSITORY_REF])
local_changes = subprocess.check_output(
    ["git", "status", "--porcelain"], cwd=REPO, text=True
).strip()
if local_changes:
    print("Saving local changes before updating:\n" + local_changes, flush=True)
    run(
        [
            "git",
            "-c",
            "user.name=SLLM Colab",
            "-c",
            "user.email=colab@localhost",
            "stash",
            "push",
            "--include-untracked",
            "--message",
            "sllm-colab-before-update",
        ]
    )
    backup = subprocess.check_output(
        ["git", "rev-parse", "refs/stash"], cwd=REPO, text=True
    ).strip()
    print(
        f"Stash backup: {backup}\n"
        f"Inspect: git stash show --include-untracked --stat {backup}\n"
        f"Restore later: git stash apply --index {backup}",
        flush=True,
    )
    remaining = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO, text=True
    ).strip()
    if remaining:
        raise RuntimeError(
            "Some changes could not be stashed; revision unchanged. Preserve these files:\n"
            + remaining
        )
run(["git", "checkout", "--detach", "FETCH_HEAD"])
print(
    "Code commit:",
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
)

## 2. Install and verify the pinned CUDA training stack

Setup checks the allocated GPU and driver with `nvidia-smi`, then installs a fully
isolated environment: PyTorch **2.14.1+cu126**, CUDA **12.6**, Accelerate **1.15.0**,
Transformers **5.20.2**, PEFT **0.21.2**. It does not inherit Colab's Python packages.
The NVIDIA driver must be at least **560.28.03** for this conservative profile.

Dependency installation uses the dedicated profile lock and package hashes. A real
tiny LoRA forward/backward/optimizer step through Accelerate checks the GPU stack;
QLoRA also checks an NF4 backward pass. No pretrained weights are downloaded by the
check. The notebook kernel stays unchanged, so no kernel restart is required.

If upgrading an older copy, reopen this updated notebook and rerun the fetch/setup
cells. The old assertion against the notebook kernel's Torch is intentionally gone.


In [ ]:
bootstrap = [sys.executable, "training/colab/bootstrap.py", "--environment", PYTHON.parent.parent]
if USE_QLORA:
    bootstrap.append("--qlora")
run(bootstrap)

## 3. Mount persistent storage

Keep model caches and raw downloads on `/content`, and checkpoints/results in
Drive. Drive access requires your own authorization. With Drive disabled, all
artifacts are ephemeral and can disappear when the runtime is deleted.


In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    STORAGE = Path("/content/drive/MyDrive/masters-sllms-research")
else:
    STORAGE = Path("/content/sllm-ephemeral")
STORAGE.mkdir(parents=True, exist_ok=True)
DATA = STORAGE / "datasets/productivity-v0"
RUN = STORAGE / "runs" / RUN_ID
print("Data:", DATA, "\nRun:", RUN)

## 4. Acquire MASSIVE 1.1 and audit the draft English productivity subset

The official archive is downloaded once per fresh VM, unless the audited data is
already saved in Drive. Only train/dev targets are materialized. Test text and test
targets are not written. The candidate allowlist and audit are **not frozen**.
Review `manual-review.train-dev.jsonl` and finish the documented data audits before
trials or final training. Original annotations that fail round-trip checks require
review; do not skip or fabricate them.


In [ ]:
import hashlib
import shutil

if not DATA.exists():
    run([PYTHON, "training/colab/download_massive.py", "--output", "data/raw/massive-1.1"])
    run(
        [
            PYTHON,
            "-m",
            "sllm_research.cli",
            "audit-massive",
            "--input",
            "data/raw/massive-1.1/en-US.jsonl",
            "--config",
            "configs/data/massive.yaml",
            "--output",
            DATA,
        ]
    )
    provenance = DATA / "source-provenance"
    provenance.mkdir()
    for item in (REPO / "data/raw/massive-1.1").iterdir():
        if item.name != "en-US.jsonl" and not item.name.endswith((".tar.gz", ".part")):
            shutil.copy2(item, provenance / item.name)
audit = json.loads((DATA / "audit.json").read_text())
config_hash = hashlib.sha256((REPO / "configs/data/massive.yaml").read_bytes()).hexdigest()
assert audit["config_sha256"] == config_hash, "Config changed: use a new dataset output directory"
print(json.dumps(audit["counts"], indent=2))
assert (DATA / "train.jsonl").exists() and (DATA / "dev.jsonl").exists()
assert not (DATA / "test.jsonl").exists()

## 5. Run LoRA (or explicitly selected QLoRA)

Default: seed 17, 64 training examples, 16 development examples, two optimizer
steps. Prompt and padding labels are masked; target tokens and EOS contribute to
loss. Overlong examples fail instead of being truncated. Development generation
uses the same fixed panel, reasoning disabled, greedy decoding, and the same scorer
for untuned/adapted outputs. Current JSON generation is unconstrained in both.

Checkpoints contain optimizer/scheduler/RNG state and a checksum completion marker.
Only completed checkpoints are eligible for `RESUME=True`. A reconnect restores the
same run, recipe, seed, data hashes, and code commit. If the VM died before the
first checkpoint, start a new run ID. Every independent experiment needs a new ID.


In [ ]:
recipe = json.loads(
    subprocess.check_output(
        [
            str(PYTHON),
            "-c",
            "import json,sys,yaml; print(json.dumps(yaml.safe_load(open(sys.argv[1]))))",
            str(REPO / RECIPE),
        ],
        text=True,
    )
)
recipe["method"] = "qlora" if USE_QLORA else "lora"
RUN_RECIPE = Path("/content/sllm-run-recipe.json")
RUN_RECIPE.write_text(json.dumps(recipe, indent=2))
command = [
    PYTHON,
    "-m",
    "sllm_research.finetune",
    "--root",
    REPO,
    "--config",
    RUN_RECIPE,
    "--candidate",
    CANDIDATE,
    "--seed",
    SEED,
    "--train",
    DATA / "train.jsonl",
    "--dev",
    DATA / "dev.jsonl",
    "--output",
    RUN,
]
if RESUME:
    command.append("--resume")
run(command)

## 6. Inspect saved outputs

A two-step run establishes pipeline behavior, not adaptation effectiveness.
`run.json` records the effective recipe, modules/parameters, precision, code and
model revisions, template hash, data hashes, and selected example IDs. Read the
per-example logs and length audit; do not choose recipes using final test labels.


In [ ]:
complete = json.loads((RUN / "complete.json").read_text())
FINAL = RUN / complete["artifact_directory"]
print("Selected adapter:", FINAL / "adapter")
print("Untuned development:", json.loads((RUN / "untuned.dev.scores.json").read_text())["summary"])
print(
    "Adapted development:", json.loads((FINAL / "adapted.dev.scores.json").read_text())["summary"]
)
print("Selection:", complete)

## 7. Optional high-precision merge and canary check

Set `DO_MERGE=True` in the configuration cell, then rerun this cell. This reloads
original unquantized base weights even for QLoRA, saves a standalone safetensors
checkpoint and tokenizer, compares a fixed development canary set before/after
merge, and records hashes. It needs extra RAM/disk/Drive quota. If canaries diverge,
the command preserves the evidence and fails for investigation.

GGUF conversion, Q4/Q8 quantization, and execution on the Realme phone are separate
steps; a successful Colab merge does not pass G1.


In [ ]:
if DO_MERGE:
    MERGED = STORAGE / "merged" / RUN_ID
    run(
        [
            PYTHON,
            "-m",
            "sllm_research.merge_adapter",
            "--run",
            RUN,
            "--dev",
            DATA / "dev.jsonl",
            "--output",
            MERGED,
        ]
    )
    print("Merged checkpoint:", MERGED)
else:
    print("Merge skipped. Enable DO_MERGE after inspecting the run.")

## 8. Reconnect, expand to full training, and hand off

- **Reconnect:** select the same runtime version; rerun setup/mount cells; set the
  exact saved code commit, original `RUN_ID`, candidate, seed, recipe, and
  `RESUME=True`. Keep the original dataset directory. Completed runs cannot resume.
- **Out of memory:** first lower micro-batch size and keep effective batch size 16
  using accumulation. The default is already micro-batch 1. Consider QLoRA as an
  explicitly recorded new run. Do not silently drop long examples or change the
  recipe of a resumed run.
- **Trial/final runs:** finish G1/G2 and record the primary model in the repository.
  Use the full recipe template, audit lengths, run at most two substantive trials,
  then freeze the recipe/G3. Run seeds 17, 42, 73 in separate directories. The entry
  point enforces gates and final seed membership.
- **Preserve artifacts:** Drive holds adapters, training checkpoints, raw development
  predictions, scores, provenance, and optional merged weights. Copy the pinned
  code/lock/config and data provenance alongside any released artifacts. No upload
  to Hugging Face or automatic real-world action execution occurs.
- **Phone handoff:** follow `export/README.md` and `android/README.md`. Benchmark
  latency/memory on the phone, not the Colab GPU.

See the complete guide for artifact layout, troubleshooting, and limitations.
